In [66]:
import torch
from sentence_transformers import SentenceTransformer
from FlagEmbedding import BGEM3FlagModel 
import numpy as np
import time
from ast import literal_eval

from pathlib import Path
import pandas as pd


DATA_PATH = Path("D:/DP/data/storysim_dataset")
titles_df = pd.read_csv(DATA_PATH / "titles.csv")
qa_df = pd.read_csv(DATA_PATH / "cleaned_qa_dataset.csv", converters={"pos_title_ids": literal_eval, "negative_title_ids": literal_eval})
assert not titles_df[titles_df["plot"].isna()].shape[0] > 0
stories = titles_df["plot"].tolist()
queries = qa_df["question"].tolist()
qa_df.describe()

,id
count,3230.000000
mean,1673.153870
std,962.808389
min,0.000000
25%,844.250000
50%,1671.500000
75%,2506.750000
max,3339.000000


In [67]:
pos_titles = qa_df["pos_title_ids"].explode()
pos_titles.groupby(pos_titles).size().sort_values(ascending=False)

def sample_queries_balance_title_ids(qa_df, n=1000):
    """
    Sample n queries from qa_df with balanced representation of title_ids.
    
    Args:
        qa_df: DataFrame with 'pos_title_ids' column containing lists of relevant title IDs
        n: Number of queries to sample
    
    Returns:
        DataFrame with sampled queries
    """
    # Get all unique title IDs and their frequency in the dataset
    all_pos_titles = qa_df["pos_title_ids"].explode()
    title_counts = all_pos_titles.groupby(all_pos_titles.values).size().sort_values(ascending=False)
    
    # Get unique queries for each title ID
    title_to_queries = {}
    for idx, row in qa_df.iterrows():
        for title_id in row["pos_title_ids"]:
            if title_id not in title_to_queries:
                title_to_queries[title_id] = []
            title_to_queries[title_id].append(idx)
    
    # Sample queries to balance title representation
    sampled_query_indices = set()
    
    # Start with least frequent titles to ensure they're represented
    for title_id in title_counts.index[::-1]:
        if len(sampled_query_indices) >= n:
            break
            
        # Get queries for this title that haven't been sampled yet
        available_queries = [q for q in title_to_queries[title_id] if q not in sampled_query_indices]
        
        if available_queries:
            # Sample one query for this title
            sampled_idx = np.random.choice(available_queries)
            sampled_query_indices.add(sampled_idx)
    
    # If we still need more queries, sample randomly from remaining
    remaining_needed = n - len(sampled_query_indices)
    if remaining_needed > 0:
        remaining_indices = [idx for idx in qa_df.index if idx not in sampled_query_indices]
        if remaining_indices:
            additional_samples = np.random.choice(
                remaining_indices, 
                size=min(remaining_needed, len(remaining_indices)), 
                replace=False
            )
            sampled_query_indices.update(additional_samples)
    
    # Return the sampled queries
    return qa_df.loc[list(sampled_query_indices)]


df = sample_queries_balance_title_ids(qa_df, 800)
df["pos_title_ids"].explode().groupby(df["pos_title_ids"].explode()).size().sort_values(ascending=False).describe()


count    956.000000
mean       2.243724
std        1.524739
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       10.000000
Name: pos_title_ids, dtype: float64

In [68]:
# model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
# stories_lens = [sum(mask) for mask in model.tokenizer(stories, padding=True, truncation=True, max_length=20000, return_tensors="pt")["attention_mask"]]
# queries_lens = [sum(mask) for mask in model.tokenizer(queries, padding=True, truncation=True, max_length=20000, return_tensors="pt")["attention_mask"]]
query_len_df = pd.DataFrame({"query_len": queries_lens})
story_len_df = pd.DataFrame({"story_len": stories_lens})
query_len_df["query_len"] = query_len_df["query_len"].apply(lambda x: x.item())
story_len_df["story_len"] = story_len_df["story_len"].apply(lambda x: x.item())

display(query_len_df.describe())
display(story_len_df.describe())

In [69]:

models_to_test = {
    "MiniLM-L6-v2": "all-MiniLM-L6-v2",
    "MPNet-base-v2": "all-mpnet-base-v2",
    "BGE-M3": "BAAI/bge-m3",
    "E5": "intfloat/multilingual-e5-large-instruct"
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def embed_documents(model_name, model_id, stories):
    if model_name == "BGE-M3":
        model = BGEM3FlagModel(model_id, use_fp16=True if device == 'cuda' else False)
        raw_output = model.encode(stories,
                                batch_size=4,
                                max_length=512, 
                                convert_to_numpy=True)
        embeddings = raw_output['dense_vecs']
        return embeddings
    else:
        model = SentenceTransformer(model_id, device=device)
        embeddings = model.encode(stories, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    return embeddings

def get_detailed_instruct(task_description: str, query: str) -> str:
            return f'Instruct: {task_description}\nQuery: {query}'

def embed_queries(model_name, model_id, queries):
    
    if model_name == "E5":
        task = 'Given a story retrieve relevant passages that is similar to the story'

        queries = [
            get_detailed_instruct(task, q)
            for q in queries
        ]
    embeddings = embed_documents(model_name, model_id, queries)
    return embeddings

query_embeddings = {}
plot_embeddings = {}


for model_name, model_id in models_to_test.items():
    print(f"\n--- Processing model: {model_name} ---")
    plot_save_file = DATA_PATH/f'plot_embeddings_{model_name}.npy'
    if plot_save_file.exists():
        embeddings = np.load(plot_save_file)
    else:
        embeddings = embed_documents(model_name, model_id, stories)
        with open(plot_save_file, 'wb') as f:
            np.save(f, embeddings)
    
    query_save_file = DATA_PATH/f'query_embeddings_{model_name}.npy'
    if query_save_file.exists():
        qembeddings = np.load(query_save_file)
    else:
        qembeddings = embed_queries(model_name, model_id, queries)
        with open(query_save_file, 'wb') as f:
            np.save(f, qembeddings)
    
    plot_embeddings[model_name] = embeddings
    query_embeddings[model_name] = qembeddings



Using device: cuda

--- Processing model: MiniLM-L6-v2 ---

--- Processing model: MPNet-base-v2 ---

--- Processing model: BGE-M3 ---

--- Processing model: E5 ---


In [70]:
def calc_metrics(q_emb, doc_embs, qa_df, titles_df, ks=[1,5,10,20]):
    """
    Calculate retrieval metrics for query embeddings against document embeddings.
    
    Args:
        q_emb: Query embeddings, where q_emb[i] corresponds to qa_df.iloc[i]
        doc_embs: Document embeddings, where doc_embs[i] corresponds to titles_df.iloc[i]
        qa_df: DataFrame containing queries and ground truth, with pos_title_ids column
        titles_df: DataFrame containing document titles (or IDs if matching is done by index)
        ks: List of k values to calculate precision and recall at (default: [1,5,10,20])
    
    Returns:
        Dictionary of retrieval metrics with keys formatted as recall@k and precision@k
    """
    similarities = np.dot(q_emb, doc_embs.T)
    
    recall = {k: [] for k in ks}
    precision = {k: [] for k in ks}
    mrr_list = []
    ap_list = []
    
    for idx, row in qa_df.iterrows():
        # Get relevant document indices
        rel_title_ids = row['pos_title_ids'] 
        relevant_docs = []
        for title in rel_title_ids:
            try:
                relevant_docs.append(titles_df[titles_df["title"] == title].index.values[0])
            except:
                continue
        
        i = qa_df.index.get_loc(idx)
        sorted_indices = np.argsort(-similarities[i])
        
        if len(relevant_docs) == 0:
            # Handle case with no relevant documents
            for k in ks:
                recall[k].append(0.0)
                precision[k].append(0.0)
            mrr_list.append(0.0)
            ap_list.append(0.0)
            continue
    
        max_k = max(ks)
        relevant_in_topk = set(sorted_indices[:max_k]).intersection(relevant_docs)
        
        for k in ks:
            recall[k].append(len(relevant_in_topk.intersection(sorted_indices[:k])) / len(relevant_docs))
            precision[k].append(len(relevant_in_topk.intersection(sorted_indices[:k])) / float(k))
        

        first_relevant_rank = next((rank+1 for rank, doc_idx in enumerate(sorted_indices) 
                                  if doc_idx in relevant_docs), 0)
        mrr_list.append(1.0 / first_relevant_rank if first_relevant_rank else 0.0)
     
        ap_temp = 0.0
        correct_count = 0
        for rank_idx, doc_idx in enumerate(sorted_indices[:max_k]):
            if doc_idx in relevant_docs:
                correct_count += 1
                ap_temp += correct_count / (rank_idx + 1)
        ap_list.append(ap_temp / len(relevant_docs) if relevant_docs else 0.0)
    
    results = {
        'mrr': np.mean(mrr_list),
        'map': np.mean(ap_list)
    }
    for k in ks:
        results[f'recall@{k}'] = np.mean(recall[k])
        results[f'precision@{k}'] = np.mean(precision[k])
    
    return results


In [71]:
# Calculate metrics for all models
results = {}
for model_name in models_to_test.keys():
    print(f"\nEvaluating {model_name}...")
    metrics = calc_metrics(
        query_embeddings[model_name],
        plot_embeddings[model_name],
        qa_df,
        titles_df
    )
    results[model_name] = metrics
    print(f"  Recall@5: {metrics['recall@5']:.4f}")
    print(f"  Recall@10: {metrics['recall@10']:.4f}")
    print(f"  MRR: {metrics['mrr']:.4f}")

# Create results dataframe
results_df = pd.DataFrame(results).T
results_df.to_csv(DATA_PATH/"embed_comparison.csv")
print("\nModel Comparison:")
display(results_df)


Evaluating MiniLM-L6-v2...


  Recall@5: 0.4215
  Recall@10: 0.5073
  MRR: 0.5043

Evaluating MPNet-base-v2...
  Recall@5: 0.4879
  Recall@10: 0.5709
  MRR: 0.5657

Evaluating BGE-M3...
  Recall@5: 0.5153
  Recall@10: 0.6207
  MRR: 0.5980

Evaluating E5...
  Recall@5: 0.7204
  Recall@10: 0.8048
  MRR: 0.7779

Model Comparison:


,mrr,map,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10,recall@20,precision@20
MiniLM-L6-v2,0.504254,0.358851,0.191150,0.403096,0.421453,0.187183,0.507339,0.112601,0.585602,0.065557
MPNet-base-v2,0.565672,0.414388,0.219135,0.461300,0.487869,0.216285,0.570897,0.127307,0.653330,0.073158
BGE-M3,0.597980,0.442713,0.233741,0.488545,0.515268,0.226068,0.620724,0.137059,0.708810,0.078390
E5,0.777891,0.648464,0.337726,0.692260,0.720444,0.317709,0.804845,0.178978,0.870576,0.097554


In [72]:
del query_embeddings
del plot_embeddings

In [73]:
e5_instructions = {
    "setting_retrieval": "Given a story description, retrieve similar stories with similar plot, characters, setting and themes",
    "character_dynamics": "Find stories with comparable character relationships and interactions based on the described personalities and roles",
    "thematic_retrieval": "Retrieve stories that explore similar themes and moral dilemmas as those presented in the query",
    "plot_development": "Identify stories with analogous narrative structures, key events, and plot progression",
    "genre_specific": "Locate stories within the same genre that share common tropes and storytelling conventions",
    "mood_based": "Find stories that match the emotional tone and atmosphere described in the query",
    "cross_genre": "Discover stories from different genres that share similar narrative elements or plot devices",
    "continuation_search": "Retrieve stories that could serve as plausible sequels or prequels to the described narrative",
    "adaptation_candidates": "Identify stories with adaptation potential matching the described format and audience"
}

results = {}
plot_embeddings = np.load('D:/DP/data/storysim_dataset/plot_embeddings_E5.npy')
for task, instruction in e5_instructions.items():
    print(f"\n--- Processing task: {task} ---")
    save_file = DATA_PATH/f'e5_instructions_{task}.npy'
    if save_file.exists():
        embeddings = np.load(save_file)
    else:
        task_queries = [get_detailed_instruct(instruction, q) for q in queries]
        embeddings = embed_documents("E5", "intfloat/multilingual-e5-large-instruct", task_queries)
        with open(save_file, 'wb') as f:
            np.save(f, embeddings)
   
    results[task] = calc_metrics(embeddings, plot_embeddings, qa_df, titles_df)

results_instructions_df = pd.DataFrame(results).T
print("\nInstructions Comparison:")
display(results_instructions_df)

results_instructions_df.to_csv(DATA_PATH / "embed_e5_instruct_results.csv")




--- Processing task: setting_retrieval ---

--- Processing task: character_dynamics ---

--- Processing task: thematic_retrieval ---

--- Processing task: plot_development ---

--- Processing task: genre_specific ---

--- Processing task: mood_based ---

--- Processing task: cross_genre ---

--- Processing task: continuation_search ---

--- Processing task: adaptation_candidates ---

Instructions Comparison:


,mrr,map,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10,recall@20,precision@20
setting_retrieval,0.773335,0.639432,0.330038,0.687307,0.711215,0.313746,0.800597,0.178080,0.865028,0.096920
character_dynamics,0.793448,0.664419,0.343206,0.713932,0.730611,0.322539,0.816547,0.181827,0.877233,0.098421
thematic_retrieval,0.785899,0.654288,0.339870,0.705573,0.720230,0.317523,0.809031,0.179876,0.873100,0.097786
plot_development,0.648522,0.512804,0.254792,0.539319,0.586674,0.260062,0.719753,0.160279,0.820462,0.091950
genre_specific,0.597019,0.460874,0.220168,0.473684,0.538279,0.239195,0.673711,0.150402,0.797760,0.089396
mood_based,0.717542,0.578812,0.294297,0.620433,0.658603,0.290898,0.758475,0.168854,0.841844,0.094412
cross_genre,0.547881,0.414773,0.194462,0.416409,0.492835,0.219752,0.622496,0.139628,0.765748,0.085944
continuation_search,0.769839,0.641039,0.325770,0.678638,0.717575,0.316347,0.809495,0.180000,0.875068,0.098251
adaptation_candidates,0.358348,0.255316,0.098695,0.207740,0.330968,0.148978,0.479887,0.108019,0.613990,0.069381


In [74]:
results_df.to_csv(DATA_PATH/"embed_comparison.csv")
results_instructions_df.to_csv(DATA_PATH / "embed_e5_instruct_results.csv")

# NUDGE method for non-parametric embedding finetunning    

In [75]:
from nudge import NUDGEN, QueriesAnswersDict
from sklearn.model_selection import train_test_split
qembs =np.load(DATA_PATH/'e5_instructions_character_dynamics.npy')
embs = np.load(DATA_PATH/'plot_embeddings_E5.npy')

def map_title_ids_to_index_for_gt(qa_df, titles_df):
    """
    So for each query we have to map the title ids to the index of the relevant titles in the titles_df dataframe
      which is aligned with the numpy array of embeddings.
    
    Questions after train/val/test split are randomly sampled from the dataframe
      so indices are not contiguous in the respective split df.
    
    This function maps the title ids to the index of the titles_df
      for the relevant docs because their precomputed/processed embeddings are aligned with the titles_df 
    After that we can directly use the index to compare with the computed relevant docs from the similarity matrix (the indices are the same)
    """
    return [titles_df[titles_df["title"].isin(qa_df["pos_title_ids"][idx])].index.tolist() for idx in qa_df.index]

train_set = QueriesAnswersDict(
    q_embs = qembs[qa_df[qa_df["split"] == "train"].index],
    q_ans_indx = map_title_ids_to_index_for_gt(qa_df[qa_df["split"] == "train"], titles_df)
)

qa_val_df, qa_test_df = train_test_split(qa_df[qa_df["split"] == "test"], test_size=0.5, random_state=31)

val_set = QueriesAnswersDict(
    q_embs = qembs[qa_val_df.index],
    q_ans_indx = map_title_ids_to_index_for_gt(qa_val_df, titles_df)
)

test_set = QueriesAnswersDict(
    q_embs = qembs[qa_test_df.index],
    q_ans_indx = map_title_ids_to_index_for_gt(qa_test_df, titles_df)
)

e5_ft_embs = NUDGEN().finetune_embeddings(embs, train_set, val_set)
np.save(DATA_PATH/'e5_ft_embs.npy', e5_ft_embs)

results = {"nudge_ft": calc_metrics(qembs[qa_test_df.index], e5_ft_embs, qa_test_df, titles_df, ks=[1,5,10,20])}
results["baseline"] = calc_metrics(qembs[qa_test_df.index], embs, qa_test_df, titles_df, ks=[1,5,10,20])
results_df = pd.DataFrame(results).T
results_df.to_csv(DATA_PATH/"embed_e5_nudge_results.csv")
display(results_df)



Calculating G
Finding gamma


,mrr,map,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10,recall@20,precision@20
nudge_ft,0.883160,0.845067,0.399124,0.839175,0.885704,0.399588,0.901271,0.204948,0.915017,0.104433
baseline,0.781079,0.661101,0.335034,0.701031,0.732113,0.322887,0.820412,0.182062,0.879141,0.097938


In [76]:
from umap import UMAP
import random

umap = UMAP(n_components=2, random_state=42)
umap_nudge_embs = umap.fit(e5_ft_embs)
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

umap_nudge_embs_2d = umap_nudge_embs.embedding_

umap_baseline_embs = UMAP(n_components=2, random_state=42).fit_transform(embs)

plot_df = pd.DataFrame({
    'UMAP1_nudge': umap_nudge_embs_2d[:, 0],
    'UMAP2_nudge': umap_nudge_embs_2d[:, 1],
    'UMAP1_baseline': umap_baseline_embs[:, 0],
    'UMAP2_baseline': umap_baseline_embs[:, 1],
    'title': titles_df['title'].values
})

question_ids = random.sample(list(qa_test_df.index), 3)
colors = ['green', 'purple', 'orange']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df['UMAP1_nudge'],
    y=plot_df['UMAP2_nudge'],
    mode='markers',
    name='Fine-tuned',
    hovertext=plot_df['title'],
    marker=dict(size=6, opacity=0.6, color='blue')
))

fig.add_trace(go.Scatter(
    x=plot_df['UMAP1_baseline'],
    y=plot_df['UMAP2_baseline'],
    mode='markers',
    name='Baseline',
    hovertext=plot_df['title'],
    marker=dict(size=6, opacity=0.6, color='red')
))

for i, qid in enumerate(question_ids):
    pos_titles = qa_df.loc[qid, "pos_title_ids"]
    
    question_text = qa_df.loc[qid, "question"]
    
    pos_indices = titles_df[titles_df["title"].isin(pos_titles)].index.tolist()
    
    fig.add_trace(go.Scatter(
        x=plot_df.loc[pos_indices, 'UMAP1_nudge'],
        y=plot_df.loc[pos_indices, 'UMAP2_nudge'],
        mode='markers',
        name=f'Question {qid} (Fine-tuned)',
        hovertext=[f"Question {qid}: {plot_df.loc[idx, 'title']}" for idx in pos_indices],
        marker=dict(size=10, color=colors[i], symbol='circle')
    ))
    
    fig.add_trace(go.Scatter(
        x=plot_df.loc[pos_indices, 'UMAP1_baseline'],
        y=plot_df.loc[pos_indices, 'UMAP2_baseline'],
        mode='markers',
        name=f'Question {qid} (Baseline)',
        hovertext=[f"Question {qid}: {plot_df.loc[idx, 'title']}" for idx in pos_indices],
        marker=dict(size=10, color=colors[i], symbol='x')
    ))

question_annotations = []
for i, qid in enumerate(question_ids):
    question_text = qa_df.loc[qid, "question"]
    short_question = question_text[:100] + "..." if len(question_text) > 100 else question_text
    
    question_annotations.append(
        dict(
            x=0.5,
            y=-0.3 - (i * 0.05),
            xref="paper",
            yref="paper",
            text=f"Question {qid}: {short_question}",
            showarrow=False,
            font=dict(color=colors[i], size=12),
            align="center"
        )
    )

fig.update_layout(
    title='Comparison of Baseline vs Fine-tuned Embeddings with Highlighted Questions',
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    legend_title='Embedding Type',
    annotations=question_annotations,
    margin=dict(b=100)
)
fig.show()





d:\narana\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

d:\narana\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

